# Divide and Conquer

In [270]:
%run interface.ipynb

### Algorytm divide and conquer - wersja bez wizualizacji

In [271]:
def max_x_position(points):
    # znajduje punkt wysunięty najdalej na prawo
    maximal_x = points[0].x
    position = 0
    for i, point in enumerate(points):
        if point.x > maximal_x:
            maximal_x = point.x
            position = i
    
    return position

In [272]:
def min_x_position(points):
    # znajduje punkt wysunięty najdalej na lewo
    minimal_x = points[0].x
    position = 0
    for i, point in enumerate(points):
        if point.x < minimal_x:
            minimal_x = point.x
            position = i
    
    return position

In [273]:
def position(a, b, c):
    # okresla polozenie punktu wzgledem zadanej prostej
    result = (b.x - a.x) * (c.y - a.y) - (b.y - a.y) * (c.x - a.x)
    return 1 if result > 1e-12 else -1 if result < 1e-12 else 0

In [274]:
def y_val_at_x(x, a, b):
    if a.x != b.x:
        m = (b.y - a.y) / (b.x - a.x)
        y = m * (x - a.x) + a.y
        return y
    else:
        return max(a.y, b.y)

In [275]:
def upper_tangent(hull_left, hull_right):
    # Ustawianie początkowych indeksów na punktach skrajnych
    left_index = max_x_position(hull_left)
    right_index = min_x_position(hull_right)
    
    middle_x = (hull_left[left_index].x + hull_right[right_index].x) / 2
    
    r_len = len(hull_right)
    l_len = len(hull_left)
    
    y_value = y_val_at_x(middle_x, hull_left[left_index], hull_right[right_index])

    while True:
        # Znajdź górną styczną do B
        updated = False
        while y_value < y_val_at_x(middle_x, hull_left[left_index], hull_right[(right_index - 1 + r_len) % r_len]):
            y_value = y_val_at_x(middle_x, hull_left[left_index], hull_right[(right_index - 1 + r_len) % r_len])
            right_index = (right_index - 1 + r_len) % r_len
            updated = True
        
        # Znajdź górną styczną do A
        while y_value < y_val_at_x(middle_x, hull_left[(left_index + 1) % l_len], hull_right[right_index]):
            y_value = y_val_at_x(middle_x, hull_left[(left_index + 1) % l_len], hull_right[right_index])
            left_index = (left_index + 1) % l_len
            updated = True

        # Jeśli nie było żadnych zmian, przerwij pętlę
        if not updated:
            break

    return left_index, right_index

In [276]:
def lower_tangent(hull_left, hull_right):
    # Ustawianie początkowych indeksów na punktach skrajnych
    left_index = max_x_position(hull_left)
    right_index = min_x_position(hull_right)
    
    middle_x = (hull_left[left_index].x + hull_right[right_index].x) / 2
    
    r_len = len(hull_right)
    l_len = len(hull_left)

    y_value = y_val_at_x(middle_x, hull_left[left_index], hull_right[right_index])

    while True:
        # Znajdź górną styczną do B
        updated = False
        while y_value > y_val_at_x(middle_x, hull_left[left_index], hull_right[(right_index - 1 + r_len) % r_len]):
            y_value = y_val_at_x(middle_x, hull_left[left_index], hull_right[(right_index - 1 + r_len) % r_len])
            right_index = (right_index - 1 + r_len) % r_len
            updated = True
        
        # Znajdź górną styczną do A
        while y_value > y_val_at_x(middle_x, hull_left[(left_index + 1) % l_len], hull_right[right_index]):
            y_value = y_val_at_x(middle_x, hull_left[(left_index + 1) % l_len], hull_right[right_index])
            left_index = (left_index + 1) % l_len
            updated = True

        # Jeśli nie było żadnych zmian, przerwij pętlę
        if not updated:
            break

    return left_index, right_index

In [277]:
def merge_hulls(left_hull, right_hull):
    
    # znajdowanie indeksow punktow tworzacych odcinki laczace otoczki
    upper_left, upper_right = upper_tangent(left_hull, right_hull)
    lower_left, lower_right = lower_tangent(left_hull, right_hull)
    
    print(left_hull)
    print(right_hull)
    print("")
    print(upper_left, upper_right)
    print(lower_left, lower_right)
    print("")
    
    # lista punktow nalezacych do zwracanej otoczki
    result = []
    
    result += [left_hull[lower_left], right_hull[lower_right]]

    if lower_right is not upper_right:
        index = lower_right + 1
        while index % len(right_hull) != upper_right:
            result.append(right_hull[index % len(right_hull)])
            index += 1
        result.append(right_hull[upper_right])

    if upper_left is not lower_left:
        result.append(left_hull[upper_left])
        index = upper_left + 1
        while index % len(left_hull) != lower_left:
            result.append(left_hull[index % len(left_hull)])
            index += 1
    
    return result

def divide_and_conquer(points):
    
    # sortowanie puntow w pierwszej kolejnosci po wspolrzednej x, nastepnie po y
    sorted_points = sorted(points, key = lambda point: (point.x, point.y))
    
    if len(points) <= 2:
        return points
    
    # podzial zbioru punktow na prawy i lewy
    middle = len(points) // 2
    left_part = sorted_points[:middle]
    right_part = sorted_points[middle:]
    
    # wyznaczenie prawej i lewej czesci otoczki
    left_hull = divide_and_conquer(left_part)
    right_hull = divide_and_conquer(right_part)
    
    # laczenie wyznaczonych fragmentow otoczki
    return merge_hulls(left_hull, right_hull)
